In [1]:
import os
import sys
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
# Set the parent directory as the current directory
os.chdir(parent_dir)

# load annotator data

In [2]:
from rdma.utils.data import read_json_file, print_json_structure
def filter_annotations_with_context(student):
    """
    Filter out annotations that do not have context.
    """
    filtered_annotations = []
    
    for annotation in student['corrected_annotations']:
        if 'context' in annotation and annotation['context'] != "No context found.":
            filtered_annotations.append(annotation)
    return filtered_annotations

data_path = "data/medical_students_data/existing_annotations"
jeffrey = read_json_file(data_path + '/jennifer_rd_step4_eval.json')
jennifer = read_json_file(data_path + '/jennifer_rd_step4_eval.json')
lauren = read_json_file(data_path + '/lauren_rd_step4_eval.json')
mya = read_json_file(data_path + '/mya_rd_step4_eval.json')


jeffrey = filter_annotations_with_context(jeffrey)
jennifer = filter_annotations_with_context(jennifer)
lauren = filter_annotations_with_context(lauren)
mya = filter_annotations_with_context(mya)


agreement_strafication = {}
# print(len(jeffrey))
# print(jeffrey)
already_annotated_id_entities = {}
for i in range(len(jeffrey)):
    dec_jeffrey = jeffrey[i]['is_rare_disease']
    dec_jennifer = jennifer[i]['is_rare_disease']
    dec_lauren = lauren[i]['is_rare_disease']
    dec_mya = mya[i]['is_rare_disease']
    if dec_jeffrey == dec_jennifer == dec_lauren == dec_mya:
        sum = 4
    elif dec_jeffrey == dec_jennifer == dec_lauren or dec_jeffrey == dec_jennifer == dec_mya or dec_jeffrey == dec_lauren == dec_mya or dec_jennifer == dec_lauren == dec_mya:
        sum = 3
    elif dec_jeffrey == dec_jennifer or dec_jeffrey == dec_lauren or dec_jeffrey == dec_mya or dec_jennifer == dec_lauren or dec_jennifer == dec_mya or dec_lauren == dec_mya:
        sum = 2
    else:
        sum = 1
    if sum not in agreement_strafication:
        agreement_strafication[sum] = []
    agreement_strafication[sum].append(jeffrey[i])
    if jeffrey[i]['document_id'] not in already_annotated_id_entities:
        already_annotated_id_entities[jeffrey[i]['document_id']] = [jeffrey[i]["entity"]]
    else:
        already_annotated_id_entities[jeffrey[i]['document_id']].append(jeffrey[i]["entity"])

    assert jeffrey[i]['document_id'] == jennifer[i]['document_id'] == lauren[i]['document_id'] == mya[i]['document_id'], "Document IDs do not match across students."


for key in agreement_strafication:
    print(f"Number of annotations with agreement score {key}: {len(agreement_strafication[key])}")

# print_json_structure(jeffrey, n=7)
# filter out all annotations that don't have context

    




Number of annotations with agreement score 4: 72
Number of annotations with agreement score 2: 7
Number of annotations with agreement score 3: 17


# Filter for all ones that they haven't done yet. Pass those for the other.

In [3]:
pass_path = "data/medical_students_data/LLM_passes"
step4 = read_json_file(f"{pass_path}/step4.json")
print_json_structure(step4, n=3)
print(step4.keys())

Dictionary:
  metadata (dict): 
  Dictionary:
    timestamp (str): 
    predictions_file (str): 
    ground_truth_file (str): 
    evaluation_file (str): 
    model_info (dict): 
    Dictionary:
      llm_type (str): 
      model_type (str): 
      temperature (float): 
      retriever (str): 
      retriever_model (str): 
  summary (dict): 
  Dictionary:
    timestamp (str): 
    categories (dict): 
    Dictionary:
      false_negatives (dict): 
      Dictionary:
        total (int): 
        confirmed_rare_disease_count (int): 
        confirmed_rare_disease_percentage (float): 
        flagged_for_review_count (int): 
        flagged_for_review_percentage (float): 
        ... and 1 more items
      false_positives (dict): 
      Dictionary:
        total (int): 
        confirmed_rare_disease_count (int): 
        confirmed_rare_disease_percentage (float): 
        flagged_for_review_count (int): 
        flagged_for_review_percentage (float): 
        ... and 1 more items
      tr

In [4]:
import json
from datetime import datetime

# Your existing setup
pass_path = "data/medical_students_data/LLM_passes"
annos = {}

pass1_old = read_json_file(pass_path + '/step3.json')
pass2_old = read_json_file(pass_path + '/step3_p2.json')
pass1 = read_json_file(pass_path + '/step3_p1_s2.json')
pass2 = read_json_file(pass_path + '/step3_p2_s2.json')

# Assuming all_valid_doc_ids is defined somewhere
# all_valid_doc_ids = set(...)  # You need to define this

def format_combined_annotations_to_evaluation_format(pass_data_list, already_annotated_id_entities):
    """
    Format multiple pass data into a single evaluation format structure with combined flagged_entities
    
    Args:
        pass_data_list: List of tuples [(pass_data, pass_name), ...]
        all_valid_doc_ids: Set of valid document IDs
    
    Returns:
        Dictionary in the evaluation format with combined flagged entities
    """
    
    all_flagged_entities = []
    detailed_results = []
    total_false_positives_count = 0
    
    # Process each pass
    for pass_data, pass_name in pass_data_list:
        # Process each patient/document in this pass
        for doc_id, doc_data in pass_data['results'].items():
            # Skip if document ID is in valid set
            
            # Process matched diseases for this document
            if 'matched_diseases' in doc_data and doc_data['matched_diseases']:
                for disease in doc_data['matched_diseases']:
                    # For summary flagged_entities (simplified format)
                    if doc_id in already_annotated_id_entities and disease.get('entity', '') in already_annotated_id_entities[doc_id]:
                        continue
                    flagged_entity = {
                        "entity": disease.get('original_entity', ''),
                        "document_id": doc_id,
                        "orpha_code": disease.get('orpha_id', ''),
                        "category": "false_positives",
                        "explanation": "Match Found in Orphanet as a Rare Disease"
                    }
                    all_flagged_entities.append(flagged_entity)
                    
                    # For detailed results (full format matching the structure you showed)
                    detailed_result = {
                        "entity": disease.get('original_entity', ''),
                        "context": disease.get('context', ''),  # Use context from the matched disease
                        "is_rare_disease": True,  # Since it matched Orphanet
                        "flag_for_review": True,  # Since it's not in valid docs
                        "explanation": "Match Found in Orphanet as a Rare Disease",
                        "document_id": doc_id,
                        "orpha_code": disease.get('orpha_id', ''),
                        "rd_term": disease.get('rd_term', ''),
                        "original_entity": disease.get('original_entity', ''),
                        "orpha_candidates": disease.get('top_candidates', []),  # Include the top candidates
                        "pass_source": pass_name
                    }
                    detailed_results.append(detailed_result)
                    total_false_positives_count += 1
    
    # Create the evaluation format structure with combined data
    evaluation_format = {
        "metadata": {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "predictions_file": f"{pass_path}/step3_combined.json",
            "ground_truth_file": f"{pass_path}/step3_combined.json",
            "evaluation_file": f"{pass_path}/evaluation_combined_results.json",
            "model_info": {
                "llm_type": "local",
                "model_type": "mistral_24b",
                "temperature": 0.5,
                "retriever": "sentence_transformer",
                "retriever_model": "abhinand/MedEmbed-small-v0.1"
            },
            "passes_included": [pass_name for _, pass_name in pass_data_list]
        },
        "summary": {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "categories": {
                "false_negatives": {
                    "total": 0,
                    "confirmed_rare_disease_count": 0,
                    "confirmed_rare_disease_percentage": 0.0,
                    "flagged_for_review_count": 0,
                    "flagged_for_review_percentage": 0.0,
                    "confirmation_status": {
                        "is_rare_disease": {"YES": 0, "NO": 0},
                        "flag_for_review": {"YES": 0, "NO": 0}
                    }
                },
                "false_positives": {
                    "total": total_false_positives_count,
                    "confirmed_rare_disease_count": total_false_positives_count,
                    "confirmed_rare_disease_percentage": 100.0 if total_false_positives_count > 0 else 0.0,
                    "flagged_for_review_count": total_false_positives_count,
                    "flagged_for_review_percentage": 100.0 if total_false_positives_count > 0 else 0.0,
                    "confirmation_status": {
                        "is_rare_disease": {"YES": total_false_positives_count, "NO": 0},
                        "flag_for_review": {"YES": total_false_positives_count, "NO": 0}
                    }
                },
                "true_positives": {
                    "total": 0,
                    "confirmed_rare_disease_count": 0,
                    "confirmed_rare_disease_percentage": 0.0,
                    "flagged_for_review_count": 0,
                    "flagged_for_review_percentage": 0.0,
                    "confirmation_status": {
                        "is_rare_disease": {"YES": 0, "NO": 0},
                        "flag_for_review": {"YES": 0, "NO": 0}
                    }
                }
            },
            "total_entities": total_false_positives_count,
            "total_flagged_for_review": total_false_positives_count,
            "flagged_for_review_percentage": 100.0 if total_false_positives_count > 0 else 0.0,
            "flagged_entities": all_flagged_entities
        },
        "results": {
            "false_negatives": [],  # Empty for now
            "false_positives": detailed_results,  # All our flagged entities go here
            "true_positives": []  # Empty for now
        }
    }
    
    return evaluation_format

# Example usage (you need to define all_valid_doc_ids first):
# all_valid_doc_ids = set(['valid_id_1', 'valid_id_2', ...])  # Define your valid IDs

# Combine both passes into a single evaluation format
pass_data_list = [(pass1, "pass1"), (pass2, "pass2")]
combined_annotations = format_combined_annotations_to_evaluation_format(pass_data_list, already_annotated_id_entities)

# Save the results
def save_annotations(annos, output_path):
    """Save annotations to JSON file"""
    with open(output_path, 'w') as f:
        json.dump(annos, f, indent=2)

# Save the combined results
# combined_annotations = format_combined_annotations_to_evaluation_format(pass_data_list, all_valid_doc_ids)
save_annotations(combined_annotations, pass_path + '/step3_combined_p2.json')

print("Annotation formatting code completed!")
print("Use format_combined_annotations_to_evaluation_format() to process both passes together.")
print("Remember to define 'all_valid_doc_ids' before running the formatting functions.")

# Complete example usage:
"""
# 1. Define your valid document IDs
all_valid_doc_ids = set(['doc1', 'doc2', ...])  # Your valid IDs here

# 2. Create list of passes to process
pass_data_list = [(pass1, "pass1"), (pass2, "pass2")]

# 3. Format combined data
combined_annotations = format_combined_annotations_to_evaluation_format(pass_data_list, all_valid_doc_ids)

# 4. Save results
save_annotations(combined_annotations, pass_path + '/step3_combined.json')
"""

Annotation formatting code completed!
Use format_combined_annotations_to_evaluation_format() to process both passes together.
Remember to define 'all_valid_doc_ids' before running the formatting functions.


'\n# 1. Define your valid document IDs\nall_valid_doc_ids = set([\'doc1\', \'doc2\', ...])  # Your valid IDs here\n\n# 2. Create list of passes to process\npass_data_list = [(pass1, "pass1"), (pass2, "pass2")]\n\n# 3. Format combined data\ncombined_annotations = format_combined_annotations_to_evaluation_format(pass_data_list, all_valid_doc_ids)\n\n# 4. Save results\nsave_annotations(combined_annotations, pass_path + \'/step3_combined.json\')\n'

In [7]:
import json
from datetime import datetime


# Your existing setup
pass_path = "data/medical_students_data/LLM_passes"
annos = {}

pass1_old = read_json_file(pass_path + '/step3.json')
pass2_old = read_json_file(pass_path + '/step3_p2.json')
pass1 = read_json_file(pass_path + '/step3_p1_s2.json')
pass2 = read_json_file(pass_path + '/step3_p2_s2.json')

# Assuming all_valid_doc_ids is defined somewhere
# all_valid_doc_ids = set(...)  # You need to define this

def format_combined_annotations_to_evaluation_format(pass_data_list, already_annotated_id_entities, previous_passes=None):
    """
    Format multiple pass data into a single evaluation format structure with combined flagged_entities
    
    Args:
        pass_data_list: List of tuples [(pass_data, pass_name), ...]
        already_annotated_id_entities: Dictionary of already annotated entities by document ID
        previous_passes: List of previous pass data to exclude from current processing
    
    Returns:
        Dictionary in the evaluation format with combined flagged entities
    """
    
    # Create set of document IDs from previous passes to exclude
    previous_doc_ids = set()
    if previous_passes:
        for prev_pass in previous_passes:
            if 'results' in prev_pass:
                previous_doc_ids.update(prev_pass['results'].keys())
    # print(previous_doc_ids)
    all_flagged_entities = []
    detailed_results = []
    total_false_positives_count = 0
    
    # Process each pass
    for pass_data, pass_name in pass_data_list:
        # Process each patient/document in this pass
        for doc_id, doc_data in pass_data['results'].items():
            # Skip if document ID is in previous passes
            if doc_id in previous_doc_ids:
                continue
                
            # Skip if document ID is in valid set (keeping original logic)
            # Note: The original code had this comment but no actual check
            # Add your valid doc check here if needed:
            # if doc_id in all_valid_doc_ids:
            #     continue
            
            # Process matched diseases for this document
            if 'matched_diseases' in doc_data and doc_data['matched_diseases']:
                for disease in doc_data['matched_diseases']:
                    # For summary flagged_entities (simplified format)
                    if doc_id in already_annotated_id_entities and disease.get('entity', '') in already_annotated_id_entities[doc_id]:
                        continue
                    flagged_entity = {
                        "entity": disease.get('original_entity', ''),
                        "document_id": doc_id,
                        "orpha_code": disease.get('orpha_id', ''),
                        "category": "false_positives",
                        "explanation": "Match Found in Orphanet as a Rare Disease"
                    }
                    all_flagged_entities.append(flagged_entity)
                    
                    # For detailed results (full format matching the structure you showed)
                    detailed_result = {
                        "entity": disease.get('original_entity', ''),
                        "context": disease.get('context', ''),  # Use context from the matched disease
                        "is_rare_disease": True,  # Since it matched Orphanet
                        "flag_for_review": True,  # Since it's not in valid docs
                        "explanation": "Match Found in Orphanet as a Rare Disease",
                        "document_id": doc_id,
                        "orpha_code": disease.get('orpha_id', ''),
                        "rd_term": disease.get('rd_term', ''),
                        "original_entity": disease.get('original_entity', ''),
                        "orpha_candidates": disease.get('top_candidates', []),  # Include the top candidates
                        "pass_source": pass_name
                    }
                    detailed_results.append(detailed_result)
                    total_false_positives_count += 1
    
    # Create the evaluation format structure with combined data
    evaluation_format = {
        "metadata": {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "predictions_file": f"{pass_path}/step3_combined.json",
            "ground_truth_file": f"{pass_path}/step3_combined.json",
            "evaluation_file": f"{pass_path}/evaluation_combined_results.json",
            "model_info": {
                "llm_type": "local",
                "model_type": "mistral_24b",
                "temperature": 0.5,
                "retriever": "sentence_transformer",
                "retriever_model": "abhinand/MedEmbed-small-v0.1"
            },
            "passes_included": [pass_name for _, pass_name in pass_data_list]
        },
        "summary": {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "categories": {
                "false_negatives": {
                    "total": 0,
                    "confirmed_rare_disease_count": 0,
                    "confirmed_rare_disease_percentage": 0.0,
                    "flagged_for_review_count": 0,
                    "flagged_for_review_percentage": 0.0,
                    "confirmation_status": {
                        "is_rare_disease": {"YES": 0, "NO": 0},
                        "flag_for_review": {"YES": 0, "NO": 0}
                    }
                },
                "false_positives": {
                    "total": total_false_positives_count,
                    "confirmed_rare_disease_count": total_false_positives_count,
                    "confirmed_rare_disease_percentage": 100.0 if total_false_positives_count > 0 else 0.0,
                    "flagged_for_review_count": total_false_positives_count,
                    "flagged_for_review_percentage": 100.0 if total_false_positives_count > 0 else 0.0,
                    "confirmation_status": {
                        "is_rare_disease": {"YES": total_false_positives_count, "NO": 0},
                        "flag_for_review": {"YES": total_false_positives_count, "NO": 0}
                    }
                },
                "true_positives": {
                    "total": 0,
                    "confirmed_rare_disease_count": 0,
                    "confirmed_rare_disease_percentage": 0.0,
                    "flagged_for_review_count": 0,
                    "flagged_for_review_percentage": 0.0,
                    "confirmation_status": {
                        "is_rare_disease": {"YES": 0, "NO": 0},
                        "flag_for_review": {"YES": 0, "NO": 0}
                    }
                }
            },
            "total_entities": total_false_positives_count,
            "total_flagged_for_review": total_false_positives_count,
            "flagged_for_review_percentage": 100.0 if total_false_positives_count > 0 else 0.0,
            "flagged_entities": all_flagged_entities
        },
        "results": {
            "false_negatives": [],  # Empty for now
            "false_positives": detailed_results,  # All our flagged entities go here
            "true_positives": []  # Empty for now
        }
    }
    
    return evaluation_format

# Example usage (you need to define all_valid_doc_ids first):
# all_valid_doc_ids = set(['valid_id_1', 'valid_id_2', ...])  # Define your valid IDs

# Combine both passes into a single evaluation format, excluding documents from old passes
pass_data_list = [(pass1, "pass1"), (pass2, "pass2")]
previous_passes_list = [pass1_old, pass2_old]
# previous_passes_list = None
combined_annotations = format_combined_annotations_to_evaluation_format(
    pass_data_list, 
    already_annotated_id_entities, 
    previous_passes=previous_passes_list
)

# Save the results
def save_annotations(annos, output_path):
    """Save annotations to JSON file"""
    with open(output_path, 'w') as f:
        json.dump(annos, f, indent=2)

# Save the combined results
save_annotations(combined_annotations, pass_path + '/step3_combined_p2_test.json')

print("Annotation formatting code completed!")
print("Use format_combined_annotations_to_evaluation_format() to process both passes together.")
print("Documents from previous passes will be excluded from the current processing.")

# Complete example usage:
"""
# 1. Define your valid document IDs (if needed)
all_valid_doc_ids = set(['doc1', 'doc2', ...])  # Your valid IDs here

# 2. Create list of passes to process
pass_data_list = [(pass1, "pass1"), (pass2, "pass2")]

# 3. Create list of previous passes to exclude
previous_passes_list = [pass1_old, pass2_old]

# 4. Format combined data
combined_annotations = format_combined_annotations_to_evaluation_format(
    pass_data_list, 
    already_annotated_id_entities, 
    previous_passes=previous_passes_list
)

# 5. Save results
save_annotations(combined_annotations, pass_path + '/step3_combined.json')
"""

Annotation formatting code completed!
Use format_combined_annotations_to_evaluation_format() to process both passes together.
Documents from previous passes will be excluded from the current processing.


'\n# 1. Define your valid document IDs (if needed)\nall_valid_doc_ids = set([\'doc1\', \'doc2\', ...])  # Your valid IDs here\n\n# 2. Create list of passes to process\npass_data_list = [(pass1, "pass1"), (pass2, "pass2")]\n\n# 3. Create list of previous passes to exclude\nprevious_passes_list = [pass1_old, pass2_old]\n\n# 4. Format combined data\ncombined_annotations = format_combined_annotations_to_evaluation_format(\n    pass_data_list, \n    already_annotated_id_entities, \n    previous_passes=previous_passes_list\n)\n\n# 5. Save results\nsave_annotations(combined_annotations, pass_path + \'/step3_combined.json\')\n'